# Best Configuration Analysis

This notebook analyzes the aggregated hyperparameter search results to find the best configuration for each combination of:
- **Model**: gcn, gin, gatv2, gatv4, sage, mlp, sagn, chebnet
- **Dataset**: parkinsons, addneuromed, motrpac
- **Node Sample Ratio**: 1.0, 0.8, 0.5, 0.3
- **Readout**: OmicsReadOut, NoReadOut
- **Method**: variance, random, correlation, distance_correlation

The best configuration is selected based on **validation F1 macro** score.

Finally, bash scripts are generated to rerun these best configurations for collecting GPU stats and timing.

## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Configuration
CSV_PATH = "/scratch/aggregated_results.csv"
OUTPUT_DIR = Path("../scripts/rerun_best_configs")
OPTIMIZATION_METRIC = "summary.best_val/f1_macro"
TEST_METRIC = "summary.best_test/f1_macro"
SEEDS = [42, 123, 456]  # Seeds for rerunning
MAX_EPOCHS = 20  # Number of epochs for rerunning

print(f"CSV Path: {CSV_PATH}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Optimization Metric: {OPTIMIZATION_METRIC}")
print(f"Seeds for rerun: {SEEDS}")
print(f"Max epochs: {MAX_EPOCHS}")

## 2. Load and Clean Data

In [ ]:
# Load the CSV
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns")

# Function to clean __val__: prefix and __NaN__ strings
def clean_value(x):
    """Clean __val__: prefix and handle __NaN__ strings."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        if x == "__NaN__":
            return np.nan
        if x.startswith("__val__:"):
            val = x.replace("__val__:", "").strip()
            # Remove surrounding quotes if present
            if val.startswith("'") and val.endswith("'"):
                val = val[1:-1]
            elif val.startswith('"') and val.endswith('"'):
                val = val[1:-1]
            return val
    return x

# Apply cleaning to all columns
print("Cleaning data...")
for col in df.columns:
    df[col] = df[col].apply(clean_value)

print("Data cleaned!")
df.head()

## 3. Define Key Columns

In [ ]:
# Grouping columns
GROUP_COLS = {
    "model": "model.model_name",
    "dataset": "dataset.loader.parameters.data_name",
    "method": "dataset.loader.parameters.method",
    "node_sample_ratio": "dataset.loader.parameters.node_sample_ratio",
    "readout": "model.readout.readout_name",
}

# Hyperparameter columns to extract for bash scripts
# Note: adjacency_threshold is handled separately in command generation
HYPERPARAM_COLS = {
    "optimizer.parameters.lr": "optimizer.parameters.lr",
    "optimizer.parameters.weight_decay": "optimizer.parameters.weight_decay",
    "model.backbone.dropout": "model.backbone.dropout",
    "model.backbone.num_layers": "model.backbone.num_layers",
    "model.backbone.hidden_channels": "model.backbone.hidden_channels",
    "model.feature_encoder.out_channels": "model.feature_encoder.out_channels",
    "model.backbone.heads": "model.backbone.heads",
    "model.backbone.num_heads": "model.backbone.num_heads",
    "model.backbone.use_layer_norm": "model.backbone.use_layer_norm",
    "model.backbone.norm": "model.backbone.norm",
    "model.readout.fc_dim": "model.readout.fc_dim",
    "model.readout.fc_dropout": "model.readout.fc_dropout",
}

# Model parameters column
PARAMS_COL = "model/params/trainable"

# Metric columns for display
METRIC_COLS = [
    "summary.best_val/f1_macro",
    "summary.best_val/f1_macro_std",
    "summary.best_val/f1_weighted",
    "summary.best_val/f1_weighted_std",
    "summary.best_val/accuracy",
    "summary.best_val/accuracy_std",
    "summary.best_val/auroc",
    "summary.best_val/auroc_std",
    "summary.best_test/f1_macro",
    "summary.best_test/f1_macro_std",
    "summary.best_test/f1_weighted",
    "summary.best_test/f1_weighted_std",
    "summary.best_test/accuracy",
    "summary.best_test/accuracy_std",
    "summary.best_test/auroc",
    "summary.best_test/auroc_std",
]

# Check unique values for grouping columns
print("=== Unique Values per Grouping Column ===")
for name, col in GROUP_COLS.items():
    unique_vals = df[col].dropna().unique()
    print(f"{name}: {list(unique_vals)}")

## 4. Find Best Configurations

In [ ]:
7*3*4*4*2+3*4*4

In [ ]:
# Convert metric column to numeric
df[OPTIMIZATION_METRIC] = pd.to_numeric(df[OPTIMIZATION_METRIC], errors="coerce")

# Filter out rows with missing optimization metric
df_valid = df[df[OPTIMIZATION_METRIC].notna()].copy()
print(f"Rows with valid {OPTIMIZATION_METRIC}: {len(df_valid)} / {len(df)}")

# Filter to only include configurations with exactly 3 runs (seeds)
REQUIRED_RUNS = 3
COUNT_COL = "summary.best_test/f1_macro_count"  # Use count from validation metric

if COUNT_COL in df_valid.columns:
    df_valid[COUNT_COL] = pd.to_numeric(df_valid[COUNT_COL], errors="coerce")
    before_filter = len(df_valid)
    df_valid = df_valid[df_valid[COUNT_COL] == REQUIRED_RUNS].copy()
    print(f"Rows with exactly {REQUIRED_RUNS} runs: {len(df_valid)} / {before_filter}")
else:
    print(f"Warning: Count column {COUNT_COL} not found, skipping run count filter")

# Group by key columns and find the best configuration
group_cols_list = list(GROUP_COLS.values())

# Find index of best row per group
idx_best = df_valid.groupby(group_cols_list)[OPTIMIZATION_METRIC].idxmax()

# Get best configurations
best_configs = df_valid.loc[idx_best].copy()
best_configs = best_configs.reset_index(drop=True)

print(f"\nFound {len(best_configs)} best configurations")
print(f"\nExpected combinations: {8} models x {3} datasets x {4} ratios x {2} readouts x {4} methods = {8*3*4*2*4}")

## 5. Display Best Configurations Summary

In [ ]:
# Create summary DataFrame
summary_cols = list(GROUP_COLS.values()) + [OPTIMIZATION_METRIC] + [c for c in METRIC_COLS if c != OPTIMIZATION_METRIC and c in best_configs.columns]

# Add trainable parameters if available
if PARAMS_COL in best_configs.columns:
    summary_cols.append(PARAMS_COL)

summary_df = best_configs[summary_cols].copy()

# Rename columns for readability
rename_map = {v: k for k, v in GROUP_COLS.items()}
rename_map[OPTIMIZATION_METRIC] = "val_f1_macro"
rename_map[PARAMS_COL] = "trainable_params"
summary_df = summary_df.rename(columns=rename_map)

# Sort by dataset, model, method, ratio, readout
summary_df = summary_df.sort_values(["dataset", "model", "method", "node_sample_ratio", "readout"])

print(f"Best configurations summary ({len(summary_df)} total):")
summary_df.head(20)

In [ ]:
summary_df.columns

In [ ]:
# Statistics per model
print("=== Average Validation F1 Macro per Model ===")
model_stats = summary_df.groupby("model")["val_f1_macro"].agg(["mean", "std", "count"])
model_stats = model_stats.sort_values("mean", ascending=False)
display(model_stats)

print("\n=== Average Validation F1 Macro per Dataset ===")
dataset_stats = summary_df.groupby("dataset")["val_f1_macro"].agg(["mean", "std", "count"])
display(dataset_stats)

print("\n=== Average Validation F1 Macro per Readout ===")
readout_stats = summary_df.groupby("readout")["val_f1_macro"].agg(["mean", "std", "count"])
display(readout_stats)

## 6. Generate Bash Script Function

In [ ]:
# Model name mapping (CSV values to config names)
MODEL_NAME_MAP = {
    "sage": "graph_sage",
    "mlp": "mlp",
    "gatv2": "gatv2",
    "sagn": "sagn",
    "GATv4": "gatv4",
    "chebnet": "chebnet",
    "gcn": "gcn",
    "gin": "gin",
}

def format_override_value(value):
    """Format a value for Hydra override."""
    if pd.isna(value):
        return None
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        # Check if it's a list-like string
        if value.startswith("[") and value.endswith("]"):
            return value
        # Check for boolean strings
        if value.lower() == "true":
            return "true"
        if value.lower() == "false":
            return "false"
        if value.lower() == "null" or value.lower() == "none":
            return "null"
        # Check if it's a string representation of a float that's an integer (e.g., "4.0")
        try:
            float_val = float(value)
            if float_val == int(float_val):
                return str(int(float_val))
            return value
        except (ValueError, TypeError):
            return value
    if isinstance(value, float):
        # Check if it's effectively an integer
        if value == int(value):
            return str(int(value))
        return str(value)
    return str(value)


def generate_training_command(row, seed, max_epochs=500):
    """Generate a training command for a configuration row."""
    model_csv = row[GROUP_COLS["model"]]
    model = MODEL_NAME_MAP.get(model_csv, model_csv)
    dataset = row[GROUP_COLS["dataset"]]
    method = row[GROUP_COLS["method"]]
    ratio = row[GROUP_COLS["node_sample_ratio"]]
    readout = row[GROUP_COLS["readout"]]
    
    # Base command
    cmd_parts = [
        "python ogbench/run.py",
        f"model={model}",
        f"dataset={dataset}",
        f"seed={seed}",
    ]
    
    # Add dataset parameters
    cmd_parts.append(f"dataset.loader.parameters.method={method}")
    cmd_parts.append(f"dataset.loader.parameters.node_sample_ratio={ratio}")
    
    # Add adjacency threshold
    adj_threshold = row.get("dataset.loader.parameters.adjacency_threshold")
    if pd.notna(adj_threshold):
        cmd_parts.append(f"dataset.loader.parameters.adjacency_threshold={adj_threshold}")
    
    # Add readout
    cmd_parts.append(f"model.readout.readout_name={readout}")
    
    # Add hyperparameters
    for hp_key, hp_col in HYPERPARAM_COLS.items():
        if hp_col in row.index:
            value = format_override_value(row[hp_col])
            if value is not None:
                # Skip readout-specific params if NoReadOut
                if readout == "NoReadOut" and hp_key in ["model.readout.fc_dim", "model.readout.fc_dropout"]:
                    continue
                # Skip model-specific params that don't apply
                if hp_key == "model.backbone.heads" and model not in ["gatv2", "gatv4"]:
                    continue
                if hp_key == "model.backbone.num_heads" and model != "sagn":
                    continue
                if hp_key == "model.backbone.use_layer_norm" and model != "gatv4":
                    continue
                if hp_key == "model.backbone.norm" and model != "mlp":
                    continue
                if hp_key == "model.backbone.num_layers" and model in ["mlp", "gatv4"]:
                    continue  # These use hidden_channels list instead
                if hp_key == "model.feature_encoder.out_channels" and model in ["mlp", "gatv4"]:
                    continue  # These use different config
                    
                cmd_parts.append(f"{hp_key}={value}")
    
    # Add training settings
    cmd_parts.append(f"trainer.max_epochs={max_epochs}")
    
    # Add wandb tags
    tags = f"[{model},{dataset},rerun_best]"
    cmd_parts.append(f"logger.wandb.tags={tags}")
    cmd_parts.append("logger.wandb.project=bgbench_best_configs_rerun")
    
    return " \\".join(cmd_parts)


# Test with one row
test_row = best_configs.iloc[0]
print("=== Sample Training Command ===")
print(generate_training_command(test_row, seed=42, max_epochs=MAX_EPOCHS))

## 7. Generate Individual Bash Scripts

In [ ]:
def generate_config_script(row, seeds, max_epochs=500):
    """Generate a bash script for a single configuration (all seeds)."""
    model_csv = row[GROUP_COLS["model"]]
    model = MODEL_NAME_MAP.get(model_csv, model_csv)
    dataset = row[GROUP_COLS["dataset"]]
    method = row[GROUP_COLS["method"]]
    ratio = row[GROUP_COLS["node_sample_ratio"]]
    readout = row[GROUP_COLS["readout"]]
    
    # Create filename
    filename = f"{dataset}_{model}_{method}_{ratio}_{readout}.sh"
    
    # Generate script content
    lines = [
        "#!/bin/bash",
        f"# Best configuration for: {dataset} / {model} / {method} / ratio={ratio} / {readout}",
        f"# Validation F1 Macro: {row[OPTIMIZATION_METRIC]:.4f}",
        "",
        "set -e  # Exit on error",
        "",
    ]
    
    for seed in seeds:
        lines.append(f"echo \"Running seed {seed}...\"")
        lines.append(generate_training_command(row, seed, max_epochs))
        lines.append("")
    
    lines.append(f'echo "Completed all seeds for {dataset}/{model}/{method}/{ratio}/{readout}"')
    
    return filename, "\n".join(lines)


# Generate scripts for all configurations
scripts = []
for idx, row in best_configs.iterrows():
    filename, content = generate_config_script(row, SEEDS, MAX_EPOCHS)
    scripts.append((filename, content))

print(f"Generated {len(scripts)} individual scripts")
print("\n=== Sample Script ===")
print(scripts[0][0])
print("-" * 50)
print(scripts[0][1][:2000])

## 8. Generate Master Bash Script

In [ ]:
def generate_master_script(scripts, output_dir):
    """Generate a master script that uses the GPU scheduler."""
    lines = [
        "#!/bin/bash",
        "# Master script to run all best configurations using GPU scheduler",
        f"# Total configurations: {len(scripts)}",
        f"# Seeds per config: {len(SEEDS)}",
        f"# Total runs: {len(scripts) * len(SEEDS)}",
        "#",
        "# This script uses gpu_scheduler.py to distribute jobs across GPUs",
        "# with maximum 1 job per GPU for fair timing measurements.",
        "#",
        "# Usage:",
        "#   ./run_all.sh              # Run with default 8 GPUs",
        "#   ./run_all.sh --num-gpus 4 # Use only 4 GPUs",
        "#   ./run_all.sh --dry-run    # Preview jobs without running",
        "#",
        "# All arguments are passed directly to gpu_scheduler.py",
        "",
        "SCRIPT_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")\" && pwd)\"",
        "cd \"$SCRIPT_DIR\"",
        "",
        'echo "========================================"',
        'echo "Best Configuration Rerun - GPU Scheduler"',
        'echo "========================================"',
        f'echo "Configurations: {len(scripts)}"',
        f'echo "Seeds per config: {len(SEEDS)}"',
        f'echo "Total runs: {len(scripts) * len(SEEDS)}"',
        'echo "========================================"',
        'echo ""',
        "",
        "# Run the GPU scheduler with any passed arguments",
        'python gpu_scheduler.py \"$@\"',
    ]
    
    return "\n".join(lines)


master_script = generate_master_script(scripts, OUTPUT_DIR)
print("=== Master Script ===")
print(master_script)

## 9. Save Outputs

In [ ]:
# # Create output directory
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# print(f"Created output directory: {OUTPUT_DIR}")

# # Save individual scripts
# for filename, content in scripts:
#     script_path = OUTPUT_DIR / filename
#     with open(script_path, "w") as f:
#         f.write(content)
#     # Make executable
#     os.chmod(script_path, 0o755)

# print(f"Saved {len(scripts)} individual scripts")

# # Save master script
# master_path = OUTPUT_DIR / "run_all.sh"
# with open(master_path, "w") as f:
#     f.write(master_script)
# os.chmod(master_path, 0o755)
# print(f"Saved master script: {master_path}")

# Ensure gpu_scheduler.py exists in output directory
# scheduler_dst = OUTPUT_DIR / "gpu_scheduler.py"
# if scheduler_dst.exists():
#     print(f"GPU scheduler already exists: {scheduler_dst}")
# else:
#     # Try to copy from known location
#     scheduler_src = Path("../scripts/rerun_best_configs/gpu_scheduler.py")
#     if scheduler_src.exists():
#         import shutil
#         shutil.copy2(scheduler_src, scheduler_dst)
#         print(f"Copied GPU scheduler: {scheduler_dst}")
#     else:
#         print("WARNING: gpu_scheduler.py not found. Please ensure it exists in the output directory.")

# Save best configs CSV
csv_path = OUTPUT_DIR / "best_configs.csv"
best_configs.to_csv(csv_path, index=False)
print(f"Saved best configs CSV: {csv_path}")

# Save summary CSV (human-readable)
summary_path = OUTPUT_DIR / "best_configs_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary CSV: {summary_path}")

print("\n=== Done! ===")
print(f"Total configurations: {len(scripts)}")
print(f"Total runs (with {len(SEEDS)} seeds): {len(scripts) * len(SEEDS)}")
print(f"\nTo run all configurations:")
print(f"  cd {OUTPUT_DIR.resolve()}")
print(f"  ./run_all.sh              # Run with all 8 GPUs")
print(f"  ./run_all.sh --num-gpus 4 # Use only 4 GPUs")
print(f"  ./run_all.sh --dry-run    # Preview jobs")

In [ ]:
summary_df

In [ ]:
# List generated files
print("=== Generated Files ===")
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name}: {size} bytes")